In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Processed_527_datapoints.csv"
TARGET = "CO2 uptake (mmol/g)"

print(RAW_PATH)
print(RAW_PATH.exists())

C:\Users\ADHITHYA\Documents\co2-adsorption-ml\data\raw\Processed_527_datapoints.csv
True


In [3]:
df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
df.head()

Shape: (527, 11)


,Surface Area (m2/g),Total Pore Volume(cm3/g),Micropore Volume (cm3/g),C (%),H (%),N (%),O (%),S (%),Temp (°C),Pressure (bar),CO2 uptake (mmol/g)
0,1281.6,0.71,0.32,69.22,3.99,0.08,26.30,0.40,25,1.0,2.625000
1,1012.6,0.56,0.22,64.83,3.64,0.38,30.77,0.38,25,1.0,2.586364
2,1408.8,0.83,0.36,72.41,3.63,0.01,23.59,0.37,25,1.0,2.920455
3,1403.9,0.85,0.33,69.51,4.35,0.76,24.94,0.44,25,1.0,2.438636
4,841.3,0.36,NaN,68.43,0.00,31.09,0.48,0.00,25,1.0,3.230000


In [4]:
print(df.dtypes)
df.describe().T

Surface Area (m2/g)         float64
Total Pore Volume(cm3/g)    float64
Micropore Volume (cm3/g)    float64
C (%)                       float64
H (%)                       float64
N (%)                       float64
O (%)                       float64
S (%)                       float64
Temp (°C)                     int64
Pressure (bar)              float64
CO2 uptake (mmol/g)         float64
dtype: object


,count,mean,std,min,25%,50%,75%,max
Surface Area (m2/g),527.0,1488.150152,728.611596,2.48,955.500,1420.00,1960.000,3337.00000
Total Pore Volume(cm3/g),499.0,0.800020,0.448172,0.02,0.480,0.69,1.030,3.09000
Micropore Volume (cm3/g),317.0,0.516845,0.272742,0.04,0.310,0.48,0.690,1.29000
C (%),527.0,76.322676,12.591289,40.40,69.830,79.10,85.330,96.96000
H (%),527.0,1.684080,1.585595,0.00,0.000,1.50,2.410,6.01000
N (%),527.0,3.262524,3.712314,0.00,0.715,2.16,4.450,38.15000
O (%),527.0,18.173321,11.632111,0.00,9.920,15.43,22.910,54.00000
S (%),527.0,0.125351,0.930828,0.00,0.000,0.00,0.000,13.20000
Temp (°C),527.0,18.462998,11.009418,0.00,0.000,25.00,25.000,30.00000
Pressure (bar),527.0,0.866636,0.311198,0.10,1.000,1.00,1.000,1.01325


In [5]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "percent": (df.isna().mean() * 100).round(1)
})

missing[missing["missing"] > 0]

,missing,percent
Total Pore Volume(cm3/g),28,5.3
Micropore Volume (cm3/g),210,39.8


In [6]:
print("Exact duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)]

Exact duplicate rows: 1


,Surface Area (m2/g),Total Pore Volume(cm3/g),Micropore Volume (cm3/g),C (%),H (%),N (%),O (%),S (%),Temp (°C),Pressure (bar),CO2 uptake (mmol/g)
208,1636.0,0.74,0.68,54.0,3.75,2.69,39.56,0.0,25,1.0,2.86
229,1636.0,0.74,0.68,54.0,3.75,2.69,39.56,0.0,25,1.0,2.86


In [7]:
TPV = "Total Pore Volume(cm3/g)"
MPV = "Micropore Volume (cm3/g)"
elements = ["C (%)", "H (%)", "N (%)", "O (%)", "S (%)"]

element_sum = df[elements].sum(axis=1)

checks = {
    "negative_values": int((df < 0).sum().sum()),
    "mpv_greater_than_tpv": int((df[MPV] > df[TPV]).sum()),
    "element_sum_over_100": int((element_sum > 100).sum()),
    "element_sum_over_101": int((element_sum > 101).sum()),
    "max_element_sum": float(element_sum.max()),
}
checks

{'negative_values': 0,
 'mpv_greater_than_tpv': 2,
 'element_sum_over_100': 71,
 'element_sum_over_101': 5,
 'max_element_sum': 104.0}

In [8]:
print("Rows where micropore > total pore volume:")
display(df[df[MPV] > df[TPV]])

print("Rows where element sum > 101%:")
display(df.loc[element_sum > 101, elements].assign(total=element_sum))

Rows where micropore > total pore volume:


,Surface Area (m2/g),Total Pore Volume(cm3/g),Micropore Volume (cm3/g),C (%),H (%),N (%),O (%),S (%),Temp (°C),Pressure (bar),CO2 uptake (mmol/g)
88,217.0,0.12,0.13,89.6,2.6,1.0,6.8,0.0,25,1.013250,1.12
91,217.0,0.12,0.13,89.6,2.6,1.0,6.8,0.0,25,0.151987,0.50


Rows where element sum > 101%:


,C (%),H (%),N (%),O (%),S (%),total
18,93.40,0.73,0.89,6.79,0.40,102.21
19,96.96,0.18,0.99,4.51,0.34,102.98
524,66.41,0.00,8.33,25.26,4.00,104.00
525,77.93,0.00,4.94,17.13,1.12,101.12
526,80.95,0.00,3.45,15.60,1.29,101.29


In [9]:
audit = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "missing_cells": int(df.isna().sum().sum()),
    "missing_by_column": {col: int(n) for col, n in df.isna().sum().items() if n > 0},
    "exact_duplicates": int(df.duplicated().sum()),
    **checks,
}

out_path = PROJECT_ROOT / "results" / "audit_summary.json"
out_path.write_text(json.dumps(audit, indent=2))

audit

{'rows': 527,
 'columns': 11,
 'missing_cells': 238,
 'missing_by_column': {'Total Pore Volume(cm3/g)': 28,
  'Micropore Volume (cm3/g)': 210},
 'exact_duplicates': 1,
 'negative_values': 0,
 'mpv_greater_than_tpv': 2,
 'element_sum_over_100': 71,
 'element_sum_over_101': 5,
 'max_element_sum': 104.0}